# Notebook 5 — Feature Engineering

In this notebook, we build the final model features selected from the EDA results.

The main goals are to:
- create features that are available at prediction time,
- handle missing values,
- encode categorical variables,
- fit all preprocessing steps using training data only,
- apply the same fitted transformations to validation and test data,
- save the final feature tables and fitted preprocessing objects.


In [1]:
import os
import json
import joblib

import numpy as np
import pandas as pd
import holidays

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

## 1. Load the Data

Notebook 3 created the random stratified split.

We load the training, validation, and test artifacts and keep them separate throughout this notebook.

In [2]:
train = pd.read_csv(
    "../artifacts/random_split/train1.csv"
)

validation = pd.read_csv(
    "../artifacts/random_split/validation1.csv"
)

test = pd.read_csv(
    "../artifacts/random_split/test1.csv"
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67529, 28)
Validation: (14470, 28)
Test: (14471, 28)


## 2. Create Prediction-Time Features

The features created here use information that is available when the order is created.

The purchase timestamp is used to create calendar features.
The estimated delivery date is used to calculate the estimated delivery period.

Delivery timestamps and other future information are not used as model features.

In [3]:
def create_features(df):
    df = df.copy()

    # Convert date columns
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"],
        errors="coerce"
    )

    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"],
        errors="coerce"
    )

    # Purchase-time features
    df["purchase_year"] = (
        df["order_purchase_timestamp"].dt.year
    )

    df["purchase_month"] = (
        df["order_purchase_timestamp"].dt.month
    )

    df["purchase_dayofweek"] = (
        df["order_purchase_timestamp"].dt.dayofweek
    )

    df["purchase_hour"] = (
        df["order_purchase_timestamp"].dt.hour
    )

    df["purchase_dayofmonth"] = (
        df["order_purchase_timestamp"].dt.day
    )

    # Weekend indicator
    df["is_weekend"] = (
        df["purchase_dayofweek"] >= 5
    ).astype(int)

    # Brazilian holiday indicator
    years = (
        df["order_purchase_timestamp"]
        .dt.year
        .dropna()
        .astype(int)
        .unique()
    )

    brazil_holidays = holidays.Brazil(
        years=years
    )

    df["is_holiday"] = (
        df["order_purchase_timestamp"]
        .dt.date
        .isin(brazil_holidays)
    ).astype(int)

    # Estimated delivery time
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)

    return df


train_fe = create_features(train)
validation_fe = create_features(validation)
test_fe = create_features(test)

print("Train after feature engineering:", train_fe.shape)
print("Validation after feature engineering:", validation_fe.shape)
print("Test after feature engineering:", test_fe.shape)

Train after feature engineering: (67529, 36)
Validation after feature engineering: (14470, 36)
Test after feature engineering: (14471, 36)


## 3. Separate the Target

The target is kept separate from the feature data.

It will not be included in the preprocessing pipeline.

In [4]:
target = "late_delivery"

y_train = train_fe[target].copy()
y_validation = validation_fe[target].copy()
y_test = test_fe[target].copy()

print("Train target:")
print(y_train.value_counts())

print("\nValidation target:")
print(y_validation.value_counts())

print("\nTest target:")
print(y_test.value_counts())

Train target:
late_delivery
0    62051
1     5478
Name: count, dtype: int64

Validation target:
late_delivery
0    13296
1     1174
Name: count, dtype: int64

Test target:
late_delivery
0    13297
1     1174
Name: count, dtype: int64


## 4. Define the Selected Features

The following features were selected based on the EDA results in Notebook 4.

High-cardinality geographic variables such as customer city and ZIP-code prefix
will be handled using frequency encoding.

In [5]:
feature_columns = [

    # Order
    "number_of_items",
    "number_of_sellers",

    # Product
    "total_product_weight",
    "total_product_volume",
    "number_of_product_categories",

    # Financial
    "total_price",
    "total_freight",
    "number_of_payments",
    "total_payment",
    "average_installments",
    "max_installments",
    "main_payment_type",

    # Geography
    "customer_seller_distance_km",
    "customer_seller_same_state",
    "customer_state",
    "customer_city",
    "customer_zip_code_prefix",

    # Purchase time
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "purchase_dayofmonth",
    "is_weekend",
    "is_holiday",

    # Estimated delivery
    "estimated_delivery_days"
]

print("Number of selected features:", len(feature_columns))

for i, feature in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {feature}")

Number of selected features: 25
01. number_of_items
02. number_of_sellers
03. total_product_weight
04. total_product_volume
05. number_of_product_categories
06. total_price
07. total_freight
08. number_of_payments
09. total_payment
10. average_installments
11. max_installments
12. main_payment_type
13. customer_seller_distance_km
14. customer_seller_same_state
15. customer_state
16. customer_city
17. customer_zip_code_prefix
18. purchase_year
19. purchase_month
20. purchase_dayofweek
21. purchase_hour
22. purchase_dayofmonth
23. is_weekend
24. is_holiday
25. estimated_delivery_days


## 5. Validate the Selected Features

Before preprocessing, we check that all selected features are available in the training data.

In [6]:
missing_features = [
    feature
    for feature in feature_columns
    if feature not in train_fe.columns
]

print("Missing selected features:")
print(missing_features)

if len(missing_features) == 0:
    print("\nAll selected features are available.")
else:
    print("\nSome selected features are missing.")

Missing selected features:
[]

All selected features are available.


## 6. Build the Raw Feature Tables

The raw feature tables contain the selected features before encoding and missing-value handling.

In [7]:
X_train_raw = train_fe[
    feature_columns
].copy()

X_validation_raw = validation_fe[
    feature_columns
].copy()

X_test_raw = test_fe[
    feature_columns
].copy()

print("X_train:", X_train_raw.shape)
print("X_validation:", X_validation_raw.shape)
print("X_test:", X_test_raw.shape)

X_train: (67529, 25)
X_validation: (14470, 25)
X_test: (14471, 25)


## 7. Check Missing Values

Missing values will be handled later by the preprocessing pipeline.

The imputation values will be learned from the training data only.

In [8]:
missing_train = (
    X_train_raw
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values in training data:")

display(
    missing_train[
        missing_train > 0
    ]
)

print(
    "\nTotal missing values:",
    X_train_raw.isna().sum().sum()
)

Missing values in training data:


customer_seller_distance_km    328
dtype: int64


Total missing values: 328


## 8. Define the Feature Groups

Low-cardinality categorical variables are one-hot encoded.

High-cardinality geographic variables are frequency encoded.

The remaining variables are treated as numerical features.

In [9]:
onehot_features = [
    "customer_state",
    "main_payment_type"
]

frequency_features = [
    "customer_city",
    "customer_zip_code_prefix"
]

numerical_features = [
    feature
    for feature in feature_columns
    if feature not in (
        onehot_features +
        frequency_features
    )
]

print("One-hot features:")
print(onehot_features)

print("\nFrequency-encoded features:")
print(frequency_features)

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of one-hot features:", len(onehot_features))
print("Number of frequency features:", len(frequency_features))
print("Number of numerical features:", len(numerical_features))

One-hot features:
['customer_state', 'main_payment_type']

Frequency-encoded features:
['customer_city', 'customer_zip_code_prefix']

Numerical features:
['number_of_items', 'number_of_sellers', 'total_product_weight', 'total_product_volume', 'number_of_product_categories', 'total_price', 'total_freight', 'number_of_payments', 'total_payment', 'average_installments', 'max_installments', 'customer_seller_distance_km', 'customer_seller_same_state', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'purchase_dayofmonth', 'is_weekend', 'is_holiday', 'estimated_delivery_days']

Number of one-hot features: 2
Number of frequency features: 2
Number of numerical features: 21


## 9. Create the Frequency Encoder

Customer city and ZIP-code prefix contain many unique values.

Instead of creating a large number of one-hot columns, we represent each category by its frequency.

The frequency values are learned from the training data only.

In [10]:
class FrequencyEncoder(
    BaseEstimator,
    TransformerMixin
):

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()

        self.feature_names_in_ = [
            str(column)
            for column in X.columns
        ]

        self.frequency_maps_ = {}

        for column in X.columns:

            values = X[column].copy()

            values = values.astype(object)
            values = values.where(
                values.notna(),
                "__MISSING__"
            )

            self.frequency_maps_[column] = (
                values.value_counts()
                .to_dict()
            )

        return self

    def transform(self, X):
        X = pd.DataFrame(
            X,
            columns=self.feature_names_in_
        ).copy()

        output = pd.DataFrame(
            index=X.index
        )

        for column in X.columns:

            values = X[column].copy()

            values = values.astype(object)
            values = values.where(
                values.notna(),
                "__MISSING__"
            )

            output[
                f"{column}_frequency"
            ] = (
                values
                .map(
                    self.frequency_maps_[column]
                )
                .fillna(0)
            )

        return output

    def get_feature_names_out(
        self,
        input_features=None
    ):
        if input_features is None:
            input_features = (
                self.feature_names_in_
            )

        return np.array(
            [
                f"{feature}_frequency"
                for feature in input_features
            ],
            dtype=object
        )

## 10. Build the Preprocessing Pipelines

Numerical missing values are replaced using the median learned from the training data.

Categorical variables are filled using the most frequent training value and then one-hot encoded.

In [11]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

onehot_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5,
                sparse_output=False
            )
        )
    ]
)

frequency_pipeline = Pipeline(
    steps=[
        (
            "frequency_encoder",
            FrequencyEncoder()
        )
    ]
)

## 11. Combine the Preprocessing Steps

ColumnTransformer applies each preprocessing step to its corresponding group of features.

Scaling is not used here because the main model is a tree-based model.

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numerical_features
        ),
        (
            "onehot",
            onehot_pipeline,
            onehot_features
        ),
        (
            "frequency",
            frequency_pipeline,
            frequency_features
        )
    ]
)

print(
    "Preprocessing pipeline created successfully."
)

Preprocessing pipeline created successfully.


## 12. Fit the Preprocessor on Training Data Only

This is the main step for preventing data leakage.

The preprocessing pipeline is fitted only on the training data.
Validation and test data are transformed using the same fitted pipeline.

In [13]:
X_train_transformed = (
    preprocessor.fit_transform(
        X_train_raw
    )
)

X_validation_transformed = (
    preprocessor.transform(
        X_validation_raw
    )
)

X_test_transformed = (
    preprocessor.transform(
        X_test_raw
    )
)

print(
    "Train transformed:",
    X_train_transformed.shape
)

print(
    "Validation transformed:",
    X_validation_transformed.shape
)

print(
    "Test transformed:",
    X_test_transformed.shape
)

Train transformed: (67529, 54)
Validation transformed: (14470, 54)
Test transformed: (14471, 54)


## 13. Get the Final Feature Names

The preprocessing step creates the final feature columns that will be used by the machine learning model.

In [14]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Number of final features:",
    len(feature_names)
)

print("\nFirst 50 final features:")

for i, feature in enumerate(
    feature_names[:50],
    start=1
):
    print(
        f"{i:02d}. {feature}"
    )

Number of final features: 54

First 50 final features:
01. num__number_of_items
02. num__number_of_sellers
03. num__total_product_weight
04. num__total_product_volume
05. num__number_of_product_categories
06. num__total_price
07. num__total_freight
08. num__number_of_payments
09. num__total_payment
10. num__average_installments
11. num__max_installments
12. num__customer_seller_distance_km
13. num__customer_seller_same_state
14. num__purchase_year
15. num__purchase_month
16. num__purchase_dayofweek
17. num__purchase_hour
18. num__purchase_dayofmonth
19. num__is_weekend
20. num__is_holiday
21. num__estimated_delivery_days
22. onehot__customer_state_AC
23. onehot__customer_state_AL
24. onehot__customer_state_AM
25. onehot__customer_state_AP
26. onehot__customer_state_BA
27. onehot__customer_state_CE
28. onehot__customer_state_DF
29. onehot__customer_state_ES
30. onehot__customer_state_GO
31. onehot__customer_state_MA
32. onehot__customer_state_MG
33. onehot__customer_state_MS
34. onehot_

## 14. Convert the Transformed Data to DataFrames

The transformed data is converted to DataFrames so it can be inspected and saved as model-ready feature tables.

In [15]:
X_train_final = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

X_validation_final = pd.DataFrame(
    X_validation_transformed,
    columns=feature_names
)

X_test_final = pd.DataFrame(
    X_test_transformed,
    columns=feature_names
)

print(
    "Final train:",
    X_train_final.shape
)

print(
    "Final validation:",
    X_validation_final.shape
)

print(
    "Final test:",
    X_test_final.shape
)

Final train: (67529, 54)
Final validation: (14470, 54)
Final test: (14471, 54)


## 15. Clean the Final Feature Names

Some generated feature names contain characters that may be inconvenient for machine learning libraries.

We replace them with letters, numbers, and underscores.

In [16]:
import re


def clean_feature_name(name):
    name = str(name)

    name = re.sub(
        r"[^A-Za-z0-9_]+",
        "_",
        name
    )

    name = re.sub(
        r"_+",
        "_",
        name
    )

    return name.strip("_")


clean_feature_names = [
    clean_feature_name(feature)
    for feature in feature_names
]

print("Feature name examples:")

for old, new in zip(
    feature_names[:20],
    clean_feature_names[:20]
):
    print(
        f"{old}  ->  {new}"
    )

Feature name examples:
num__number_of_items  ->  num_number_of_items
num__number_of_sellers  ->  num_number_of_sellers
num__total_product_weight  ->  num_total_product_weight
num__total_product_volume  ->  num_total_product_volume
num__number_of_product_categories  ->  num_number_of_product_categories
num__total_price  ->  num_total_price
num__total_freight  ->  num_total_freight
num__number_of_payments  ->  num_number_of_payments
num__total_payment  ->  num_total_payment
num__average_installments  ->  num_average_installments
num__max_installments  ->  num_max_installments
num__customer_seller_distance_km  ->  num_customer_seller_distance_km
num__customer_seller_same_state  ->  num_customer_seller_same_state
num__purchase_year  ->  num_purchase_year
num__purchase_month  ->  num_purchase_month
num__purchase_dayofweek  ->  num_purchase_dayofweek
num__purchase_hour  ->  num_purchase_hour
num__purchase_dayofmonth  ->  num_purchase_dayofmonth
num__is_weekend  ->  num_is_weekend
num__is_hol

## 16. Check for Duplicate Feature Names

Cleaning the feature names should not create duplicate columns.

In [17]:
duplicate_names = (
    pd.Series(
        clean_feature_names
    )
    .duplicated()
)

print(
    "Duplicate cleaned names:",
    duplicate_names.sum()
)

if duplicate_names.sum() == 0:
    print(
        "All cleaned feature names are unique."
    )
else:
    print(
        "Duplicate feature names were found."
    )

Duplicate cleaned names: 0
All cleaned feature names are unique.


## 17. Apply the Cleaned Feature Names

In [18]:
X_train_final.columns = (
    clean_feature_names
)

X_validation_final.columns = (
    clean_feature_names
)

X_test_final.columns = (
    clean_feature_names
)

print("First 30 cleaned feature names:")

for feature in (
    X_train_final.columns[:30]
):
    print(feature)

First 30 cleaned feature names:
num_number_of_items
num_number_of_sellers
num_total_product_weight
num_total_product_volume
num_number_of_product_categories
num_total_price
num_total_freight
num_number_of_payments
num_total_payment
num_average_installments
num_max_installments
num_customer_seller_distance_km
num_customer_seller_same_state
num_purchase_year
num_purchase_month
num_purchase_dayofweek
num_purchase_hour
num_purchase_dayofmonth
num_is_weekend
num_is_holiday
num_estimated_delivery_days
onehot_customer_state_AC
onehot_customer_state_AL
onehot_customer_state_AM
onehot_customer_state_AP
onehot_customer_state_BA
onehot_customer_state_CE
onehot_customer_state_DF
onehot_customer_state_ES
onehot_customer_state_GO


## 18. Check for Data Leakage

The final feature set must not contain the target or information that becomes available only after the prediction point.

In [19]:
forbidden_features = [

    # Target and target-derived information
    "late_delivery",
    "delivery_delay_days",
    "delivery_duration_days",

    # Post-delivery information
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "carrier_handling_days",
    "carrier_to_customer_days",
    "purchase_to_carrier_days",
    "delivery_time_difference",

    # Approval information
    "order_approved_at",
    "approval_delay_hours",
    "purchase_to_approval_days",
    "purchase_to_approval_hours",
    "approval_delay_group",

    # IDs and non-model columns
    "order_id",
    "customer_id",
    "customer_unique_id",
    "seller_id",
    "product_id",
    "order_status",
    "order_status_code",

    # Previously rejected feature
    "zip_code_region"
]

present_forbidden = [
    feature
    for feature in feature_columns
    if feature in forbidden_features
]

target_in_features = (
    target in feature_columns
)

print("Forbidden features found:")
print(present_forbidden)

print(
    "\nTarget inside feature columns:",
    target_in_features
)

if (
    not present_forbidden
    and not target_in_features
):
    print(
        "\nNo known leakage features are included."
    )
else:
    print(
        "\nPotential leakage detected."
    )

Forbidden features found:
[]

Target inside feature columns: False

No known leakage features are included.


## 19. Final Data Quality Checks

The preprocessing pipeline should remove all missing values.

The three datasets must also contain exactly the same feature columns.

In [21]:
train_nan = (
    X_train_final
    .isna()
    .sum()
    .sum()
)

validation_nan = (
    X_validation_final
    .isna()
    .sum()
    .sum()
)

test_nan = (
    X_test_final
    .isna()
    .sum()
    .sum()
)

print(
    "Train NaN:",
    train_nan
)

print(
    "Validation NaN:",
    validation_nan
)

print(
    "Test NaN:",
    test_nan
)

same_train_validation = (
    list(X_train_final.columns)
    ==
    list(X_validation_final.columns)
)

same_train_test = (
    list(X_train_final.columns)
    ==
    list(X_test_final.columns)
)

print(
    "\nTrain == Validation:",
    same_train_validation
)

print(
    "Train == Test:",
    same_train_test
)

Train NaN: 0
Validation NaN: 0
Test NaN: 0

Train == Validation: True
Train == Test: True


## 20. Save the Final Feature Tables

The processed feature tables and target files are saved separately for use in the next notebook.

In [22]:
artifact_dir = (
    "../artifacts/random_split/notebook5"
)

os.makedirs(
    artifact_dir,
    exist_ok=True
)

X_train_final.to_csv(
    f"{artifact_dir}/train_features1.csv",
    index=False
)

X_validation_final.to_csv(
    f"{artifact_dir}/validation_features1.csv",
    index=False
)

X_test_final.to_csv(
    f"{artifact_dir}/test_features1.csv",
    index=False
)

y_train.to_csv(
    f"{artifact_dir}/train_target1.csv",
    index=False
)

y_validation.to_csv(
    f"{artifact_dir}/validation_target1.csv",
    index=False
)

y_test.to_csv(
    f"{artifact_dir}/test_target1.csv",
    index=False
)

print(
    "Feature tables and targets saved successfully."
)

Feature tables and targets saved successfully.


## 21. Save the Fitted Preprocessor and Feature Lists

The fitted preprocessing pipeline is saved so it can be reused later without fitting it again on new data.

The raw feature list and final processed feature list are also saved for reference.

In [23]:
joblib.dump(
    preprocessor,
    f"{artifact_dir}/preprocessor1.joblib"
)

with open(
    f"{artifact_dir}/raw_feature_list1.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        feature_columns,
        file,
        indent=4
    )

with open(
    f"{artifact_dir}/feature_list1.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        clean_feature_names,
        file,
        indent=4
    )

print(
    "Preprocessor and feature lists saved successfully."
)

Preprocessor and feature lists saved successfully.


## 22. Verify the Saved Artifacts

The saved files are checked to make sure all required Notebook 5 artifacts were created successfully.

In [24]:
saved_files = sorted(
    os.listdir(artifact_dir)
)

print("Saved artifacts:")

for file_name in saved_files:
    print(
        "-",
        file_name
    )

Saved artifacts:
- feature_list1.json
- preprocessor1.joblib
- raw_feature_list1.json
- test_features1.csv
- test_target1.csv
- train_features1.csv
- train_target1.csv
- validation_features1.csv
- validation_target1.csv


In [25]:
print("=" * 60)
print("NOTEBOOK 5 SUMMARY")
print("=" * 60)

print(
    "Selected raw features:",
    len(feature_columns)
)

print(
    "Final processed features:",
    X_train_final.shape[1]
)

print(
    "\nTrain shape:",
    X_train_final.shape
)

print(
    "Validation shape:",
    X_validation_final.shape
)

print(
    "Test shape:",
    X_test_final.shape
)

print(
    "\nTrain late rate:",
    round(
        y_train.mean() * 100,
        2
    ),
    "%"
)

print(
    "Validation late rate:",
    round(
        y_validation.mean() * 100,
        2
    ),
    "%"
)

print(
    "Test late rate:",
    round(
        y_test.mean() * 100,
        2
    ),
    "%"
)

print(
    "\nPreprocessor fitted on training data only."
)

print(
    "Notebook 5 completed successfully."
)

NOTEBOOK 5 SUMMARY
Selected raw features: 25
Final processed features: 54

Train shape: (67529, 54)
Validation shape: (14470, 54)
Test shape: (14471, 54)

Train late rate: 8.11 %
Validation late rate: 8.11 %
Test late rate: 8.11 %

Preprocessor fitted on training data only.
Notebook 5 completed successfully.


## Final Summary

In this notebook, the selected features were transformed into a machine-learning-ready format.

Time-based features were created from the purchase timestamp, categorical variables were encoded, missing values were handled, and frequency encoding was applied to high-cardinality geographic features.

The final processed datasets contain 65 initial engineered features before the final model selection stage. The preprocessing pipeline and processed train, validation, and test datasets were saved as artifacts for Notebook 6.